### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.  

In [8]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq", temperature=0.7)

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
  model=model,
  checkpointer=InMemorySaver(),
  middleware=[
    SummarizationMiddleware(
      model=model,
      trigger=("messages", 10),
      keep=("messages",4)
    ),
  ]
)

In [6]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [9]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
  response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
  print(f"Messages: {response}")
  print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='4591ce27-4870-42bc-a4fc-0ba4330ae3aa'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks a simple math question: "What is 2+2?" Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 78, 'total_tokens': 118, 'completion_time': 0.082576883, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.003095583, 'prompt_tokens_details': None, 'queue_time': 0.158590026, 'total_time': 0.085672466}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c15aa9c1b7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04c66-7666-7fc3-bd1f-83f3a0390c99-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 40, 'total_tokens': 118, 'output_token_details': {'reasoning': 21}})]}
Me

### Token Size

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_agent(
  model=model,
  tools=[search_hotels],
  checkpointer=InMemorySaver(),
  middleware=[
    SummarizationMiddleware(
      model=model,
      trigger=("tokens", 550),
      keep=("tokens", 200)
    )
  ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
  total_chars = sum(len(str(m.content)) for m in messages)
  return total_chars // 4  # 4 chars ≈ 1 token

In [11]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
  response = agent.invoke(
    {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
    config=config
  )
  
  tokens = count_tokens(response["messages"])
  print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
  print(f"{(response['messages'])}")

Paris: ~295 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='c3b9b9ef-7d2b-4440-b36b-db514023d397'), AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants hotels in Paris. Use function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_302f702f-65de-44fa-9d74-ac61f0ef5053', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 129, 'total_tokens': 174, 'completion_time': 0.094850412, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.004816714, 'prompt_tokens_details': None, 'queue_time': 0.376165546, 'total_time': 0.099667126}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04c6b-b83a-7c22-a8ba-b254e181d280-0'

### Fraction

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
  model=model,
  tools=[search_hotels],
  checkpointer=InMemorySaver(),
  middleware=[
    SummarizationMiddleware(
      model=model,
      trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
      keep=("fraction", 0.002),     # 0.2% = ~256 tokens
    ),
  ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
  response = agent.invoke(
    {"messages": [HumanMessage(content=f"Hotels in {city}")]},
    config=config
  )
  tokens = count_tokens(response["messages"])
  fraction = tokens / 128000  # gpt-4o-mini context
  print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
  print(response['messages'])

Paris: ~260 tokens (0.2031%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='02abf8d4-3c52-4f10-913c-8dc89d86dde8'), AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants hotels in Paris. We should call search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_32f24134-7d14-46d9-83a7-cd1b8144bf47', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 120, 'total_tokens': 166, 'completion_time': 0.0967802, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.004716719, 'prompt_tokens_details': None, 'queue_time': 0.316066978, 'total_time': 0.101496919}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c652c0ffaa', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a04c6f-3938-7d32-a7e6-bc402e29ad7e-0

### Human in the loop middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
  """Mock function to read an email by its ID."""
  return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
  """Mock function to send an email."""
  return f"Email sent to {recipient} with subject '{subject}'"

In [16]:
agent=create_agent(
  model=model,
  tools=[read_email_tool,send_email_tool],
  checkpointer=InMemorySaver(),
  middleware=[
    HumanInTheLoopMiddleware(
      interrupt_on={
        "send_email_tool":{
          "allowed_decisions":["approve","edit","reject"]
        },
        "read_email_tool":False,
      }
    )
  ]
)

In [17]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [18]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='90d54f24-3aa6-4f37-adc2-3ddafec1c681'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send an email using send_email_tool. Use function.', 'tool_calls': [{'id': 'fc_fccb7ba8-24ab-4ce8-bbf3-6a583053a6b0', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 174, 'total_tokens': 235, 'completion_time': 0.128473691, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.00661666, 'prompt_tokens_details': None, 'queue_time': 0.315991147, 'total_time': 0.135090351}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 

In [19]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
  print("⏸️ Paused! Approving...")
  
  result = agent.invoke(
    Command(
      resume={
        "decisions": [
          {"type": "approve"}
        ]
      }
    ),
    config=config
  )
  
  print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been sent to **john@test.com** with the subject **“Hello”** and the body **“How are you?”**. Let me know if you need anything else!


In [20]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='90d54f24-3aa6-4f37-adc2-3ddafec1c681'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send an email using send_email_tool. Use function.', 'tool_calls': [{'id': 'fc_fccb7ba8-24ab-4ce8-bbf3-6a583053a6b0', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 174, 'total_tokens': 235, 'completion_time': 0.128473691, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.00661666, 'prompt_tokens_details': None, 'queue_time': 0.315991147, 'total_time': 0.135090351}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 

### Reject

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
  """Mock function to read an email by its ID."""
  return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
  """Mock function to send an email."""
  return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
  model=model,
  tools=[read_email_tool,send_email_tool],
  checkpointer=InMemorySaver(),
  middleware=[
    HumanInTheLoopMiddleware(
      interrupt_on={
        "send_email_tool": {
            "allowed_decisions": ["approve", "edit", "reject"],
        },
        "read_email_tool": False,
      }
    ),
  ],
)

In [23]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [24]:
# Step 2: Reject
if "__interrupt__" in result:
  print("⏸️ Paused! Approving...")
  
  result = agent.invoke(
    Command(
      resume={
        "decisions": [
          {"type": "reject"}
        ]
      }
    ),
    config=config
  )
  
  print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I’m sorry—I wasn’t able to send the email because the request to use the email‑sending tool was rejected. Would you like me to try again, or is there something else I can help you with?


In [25]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='983887f2-c90b-4839-bbc1-28b93ad8beba'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_0d898aaa-2891-4f39-9203-cc4ce58597ba', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 174, 'total_tokens': 232, 'completion_time': 0.12013184, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.014946083, 'prompt_tokens_details': None, 'queue_time': 0.288270194, 'total_time': 0.135077923}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 

### Editing

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [33]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [34]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='52780589-d6c5-4f03-9f5a-63313867a05a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide required fields.', 'tool_calls': [{'id': 'fc_9ab7d24c-2431-4ea8-8ba8-ceaff949922f', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 172, 'total_tokens': 231, 'completion_time': 0.1272436, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.007005686, 'prompt_tokens_details': None, 'queue_time': 0.316564401, 'total_time': 0.134249286}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'mode

In [37]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

    # print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...


In [38]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='52780589-d6c5-4f03-9f5a-63313867a05a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide required fields.', 'tool_calls': [{'id': 'fc_9ab7d24c-2431-4ea8-8ba8-ceaff949922f', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 172, 'total_tokens': 231, 'completion_time': 0.1272436, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.007005686, 'prompt_tokens_details': None, 'queue_time': 0.316564401, 'total_time': 0.134249286}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'mode